In [ ]:
import os

# ── Pull GitHub Repository ──────────────────────────────────────────────────
# Only clone if we aren't already inside the repository directory
if not os.path.exists('engine.py'):
    print("Cloning repository...")
    try:
        from google.colab import userdata
        token = userdata.get('GITHUB_TOKEN')
        !git clone https://{token}@github.com/falcon978/Insurance-RAG.git --quiet
    except Exception:
        # Fallback to public clone if token isn't provided
        !git clone https://github.com/falcon978/Insurance-RAG.git --quiet
    
    # Change working directory into the repo
    %cd Insurance-RAG
    print("✅ Cloned and entered Insurance-RAG repository.")
else:
    print("✅ Repository already present.")

In [ ]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Update this path to wherever you uploaded your 'insurance-rag' folder in your Drive
PROJECT_PATH = "/content/drive/MyDrive/falcon978/insurance-rag/Insurance-RAG-ebee0665cbfaff8f853fd92902a9bc652d4c742a"

# Change directory to the project root
os.chdir(PROJECT_PATH)
print(f"Current working directory: {os.getcwd()}")

In [ ]:
# ── Install Dependencies ────────────────────────────────────────────────────
!pip install -r requirements.txt --quiet

# nest_asyncio is required for FastAPI/async routines to run inside Jupyter's event loop
import nest_asyncio
nest_asyncio.apply()

import sys

# 1. Strip the corrupted versions from the active executable
!{sys.executable} -m pip uninstall -y numpy scipy spacy scikit-learn pandas

# 2. Install the mathematically locked versions directly into the kernel
!{sys.executable} -m pip install numpy==1.26.4 scipy==1.12.0 "spacy<3.8.0" "scikit-learn<1.5.0" "pandas<2.2.0" --quiet

print('✅ Dependencies installed and async patched.')

In [ ]:
import logging
import sys

logging.basicConfig(level=logging.INFO, stream=sys.stdout)
logging.getLogger("sentence_transformers").setLevel(logging.WARNING)

In [ ]:
import os
import sys

# Ensure the notebook can import your local modules
sys.path.append(os.path.abspath('.'))

from google.colab import userdata
try:
    os.environ["GEMINI_API_KEY"] = userdata.get('GEMINI_API_KEY')
except Exception:
    os.environ["GEMINI_API_KEY"] = input("Enter your Google Gemini API Key: ")

# 2. Configure Pinecone
os.environ["VECTOR_DB_TYPE"] = "pinecone" # <--- This tells config.py to switch
os.environ["PINECONE_INDEX_NAME"] = "health-insurance-policies" # Replace with your actual Pinecone index name
os.environ["HF_DEVICE"] = "cuda"

try:
    os.environ["PINECONE_API_KEY"] = userdata.get('PINECONE_API_KEY')
except Exception:
    os.environ["PINECONE_API_KEY"] = input("Enter your Pinecone API Key: ")

# 3. Configure LangSmith Tracing
os.environ["LANGSMITH_TRACING_V2"] = "true"
os.environ["LANGSMITH_PROJECT"] = "Insurance-RAG" # Change this to whatever project name you prefer

try:
    os.environ["LANGSMITH_API_KEY"] = userdata.get('LANGSMITH_API_KEY')
except Exception:
    os.environ["LANGSMITH_API_KEY"] = input("Enter your LangSmith API Key: ")

#4 Configure DeepEval
os.environ["DEEPEVAL_RESULTS_FOLDER"] = "./eval_results"
os.environ["DEEPEVAL_PER_TASK_TIMEOUT_SECONDS_OVERRIDE"] = "1000"
os.environ["JUDGE_MODEL_NAME"] = "llama-3.3-70b-versatile" # Ensure DeepEval uses models available on GROQ console
try:
    os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')
except Exception:
    os.environ["GROQ_API_KEY"] = input("Enter your GROQ API Key: ")

try:
    os.environ["CONFIDENT_API_KEY"] = userdata.get('CONFIDENT_API_KEY')
except Exception:
    os.environ["CONFIDENT_API_KEY"] = input("Enter your Confident API Key: ")

print("✅ Environment variables configured.")

In [ ]:
%cd /content/Insurance-RAG
!git pull origin main

In [ ]:
import importlib

# 1. Import the base modules themselves (required for importlib to target them)
import rag.generator
import rag_ingestion.pipeline

# 2. Force Python to flush the cache and reload the files from the hard drive
importlib.reload(rag.generator)
importlib.reload(rag_ingestion.pipeline)

# 3. Re-import the specific classes you are actually using in the notebook
from rag.generator import ResponseGenerator
from rag_ingestion.pipeline import ExtractionPipeline

print("✅ Modules manually reloaded! Notebook is now using the latest code.")

In [ ]:
import urllib.request
from pathlib import Path

PDFS = {
    "optima_secure": {
        "name"  : "HDFC Ergo Optima Secure",
        "url"   : (
            "https://customer-portal-assets.hdfcergo.com/assets/v2/docs/"
            "default-source/downloads/policy-wordings/health/"
            "optima-secure-revision/optima-secure-revision-pw-647504209314.pdf"
        ),
        "path"  : "hdfc_optima_secure.pdf",
    },
    "care_supreme": {
        "name"  : "Care Insurance Supreme",
        "url"   : (
            "https://cms.careinsurance.com/cms/public/uploads/download_center/"
            "care-supreme---policy-terms-&-conditions-(effective-from-19-march-2025).pdf"
            "?rv=0.86869200%201775054695"
        ),
        "path"  : "care_supreme.pdf",
    },
}

for key, info in PDFS.items():
    if not Path(info['path']).exists():
        print(f"Downloading {info['name']} …")
        try:
            req = urllib.request.Request(
                info['url'],
                headers={'User-Agent': 'Mozilla/5.0'}
            )
            with urllib.request.urlopen(req, timeout=60) as r:
                Path(info['path']).write_bytes(r.read())
            size = Path(info['path']).stat().st_size // 1024
            print(f"  ✅ {info['path']}  ({size} KB)")
        except Exception as e:
            print(f"  ❌ Failed: {e}")
    else:
        print(f"✅ {info['path']} already exists")

In [ ]:
import os
from config import settings
from rag_ingestion.pipeline import ExtractionPipeline

CHUNK_SIZE = 1200
OVERLAP    = 150

# 3. Check for the local BM25 caches
# We expect them in the directory defined by settings.bm25_dir
os.makedirs(settings.bm25_dir, exist_ok=True)
bm25_optima_path = os.path.join(settings.bm25_dir, "insurance_optima_secure_bm25.pkl")
bm25_care_path = os.path.join(settings.bm25_dir, "insurance_care_supreme_bm25.pkl")

if not os.path.exists(bm25_optima_path) or not os.path.exists(bm25_care_path):
    print("\nLocal BM25 cache missing (Colab restart detected). Running extraction pipeline...")
    print("Note: If Pinecone vectors exist, they will be safely skipped.\n")

    for key, info in PDFS.items():
        if not Path(info['path']).exists():
            print(f"⚠️  {info['path']} not found — skipping")
            continue

        print(f"\n{'='*60}")
        print(f"Extracting & Indexing: {info['name']}")
        print('='*60)

        # Pass the unique collection name (insurance_optima_secure, insurance_care_supreme)
        result = ExtractionPipeline(
            pdf_path        = info['path'],
            chunk_size      = CHUNK_SIZE,
            chunk_overlap   = OVERLAP,
            collection_name = f"insurance_{key}",
            device          = "cuda"  # Change to 'cuda' if GPU is available
        ).run()

        print(f"Stats: {result.stats}")
    
    print("\n✅ Ingestion & local BM25 cache hydration complete.")
else:
    print("✅ Local BM25 cache found. Proceeding to evaluation.")

In [ ]:
# Run the complete suite: Recall, Precision, Faithfulness, Relevancy, and Custom Logic Adherence
!deepeval test run evaluations/test_cases/test_standard_rag_metrics.py

In [ ]:
print("--- RUNNING RERANKER IMPACT TEST ---")
!deepeval test run evaluations/test_cases/test_reranker_impact.py

print("\n--- RUNNING HYBRID VS SEMANTIC TEST ---")
!deepeval test run evaluations/test_cases/test_hybrid_vs_semantic.py

In [ ]:
import json
import pandas as pd
from IPython.display import display

# 1. Load the DeepEval JSON dump
with open('eval_results.json', 'r') as f:
    data = json.load(f)

# 2. Flatten the nested JSON into a tabular format
rows = []
for tc in data.get('testCases', []):
    row = {
        "Query": tc.get("input", ""),
        "Overall Status": "✅ Pass" if tc.get("success") else "❌ Fail",
        "Generated Response": tc.get("actualOutput", "")[:150] + "..." # Truncate for readability
    }
    
    # Extract individual metric scores
    for metric in tc.get('metrics', []):
        metric_name = metric['name']
        row[f"{metric_name} Score"] = metric['score']
        
        # Optional: capture the reasoning for failed metrics
        if not metric['success']:
            row[f"{metric_name} Error"] = metric['reason']
            
    rows.append(row)

df = pd.DataFrame(rows)

# 3. Create a "UI-ish" visualization using Pandas Styler
def style_status(val):
    if val == '✅ Pass':
        return 'background-color: #d4edda; color: #155724; font-weight: bold;'
    elif val == '❌ Fail':
        return 'background-color: #f8d7da; color: #721c24; font-weight: bold;'
    return ''

def style_scores(val):
    if isinstance(val, (int, float)):
        if val >= 0.8: return 'color: green;'
        if val < 0.5: return 'color: red;'
    return ''

# Apply the styling and render the interactive HTML table
styled_df = df.style\
    .applymap(style_status, subset=['Overall Status'])\
    .applymap(style_scores)\
    .set_properties(**{'text-align': 'left', 'max-width': '300px', 'white-space': 'pre-wrap'})\
    .set_caption("Insurance RAG Evaluation Results")

display(styled_df)